In [3]:
import csv
import json
import os

import datasets


_CITATION = """\
@InProceedings{ealvaradob:dataset,
title = {Phishing Datasets},
author={Esteban Alvarado},
year={2024}
}
"""

_DESCRIPTION = """\
Dataset designed for phishing classification tasks in various data types.
"""
_HOMEPAGE = ""

_LICENSE = ""

_URLS = {
    "texts": "texts.json",
    "urls": "urls.json",
    "webs": "webs.json",
    "combined_full": "combined_full.json",
    "combined_reduced": "combined_reduced.json"
}

In [7]:
class PhishingDatasets(datasets.GeneratorBasedBuilder):
    """Phishing Datasets Configuration"""

    VERSION = datasets.Version("1.1.0")

    BUILDER_CONFIGS = [
        datasets.BuilderConfig(name="texts", version=VERSION, description="text subset"),
        datasets.BuilderConfig(name="urls", version=VERSION, description="urls subset"),
        datasets.BuilderConfig(name="webs", version=VERSION, description="webs subset"),
        datasets.BuilderConfig(name="combined_full", version=VERSION, description="combined dataset that have all URLs"),
        datasets.BuilderConfig(name="combined_reduced", version=VERSION, description="combined dataset that doesn't have all URLs for representativity issues"),
    ]

    DEFAULT_CONFIG_NAME = "combined_reduced"

    def _info(self):
        features = datasets.Features(
            {
                "text": datasets.Value("string"),
                "label": datasets.Value("int64"),
            }
        )
        return datasets.DatasetInfo(
            description=_DESCRIPTION,
            features=features,
            supervised_keys=("text", "label"),
            homepage=_HOMEPAGE,
            license=_LICENSE,
            citation=_CITATION,
        )

    def _split_generators(self, dl_manager):
        urls = _URLS[self.config.name]
        data_dir = dl_manager.download_and_extract(urls)
        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                gen_kwargs={
                    "filepath": data_dir,
                    "split": "train",
                },
            ),
        ]

    def _generate_examples(self, filepath, split):
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
            for index, sample in enumerate(data):
                yield index, {
                    "text": sample['text'],
                    "label": sample['label']
                }

# Przykład ładowania i wyświetlenia danych (zakładając, że pliki JSON znajdują się w Twoim katalogu roboczym):
# dataset = datasets.load_dataset("./nazwa_pliku_skryptu.py", name="combined_reduced")
# print(dataset["train"][0])

In [9]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import os

# Ponieważ zdalne skrypty datasetów są wyłączone, pobieramy bezpośrednio plik JSON z repozytorium Hugging Face:
import urllib.request
import json

url = "https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/combined_reduced.json"
response = urllib.request.urlopen(url)
data = json.loads(response.read().decode('utf-8'))

# Tworzymy obiekt Dataset bezpośrednio z pobranych danych
df_data = {"text": [item["text"] for item in data], "label": [item["label"] for item in data]}
dataset = Dataset.from_dict(df_data)

df = dataset.to_pandas()
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, test = Dataset.from_pandas(train, preserve_index=False), Dataset.from_pandas(test, preserve_index=False)

In [10]:
len(train)

62141